# 🇻🇳 PhoBERT-large × UIT-VSMEC — Vietnamese Emotion Classification

| | |
|---|---|
| **Model** | `vinai/phobert-large` |
| **Dataset** | `tridm/UIT-VSMEC` (7 classes, ~5.5k train) |
| **Platform** | Kaggle T4 (16 GB VRAM) |
| **Hub** | `qdovan03/phobert-large-uit-vsmec` |

**Flow:** Load → Normalize → Word-segment (RDRSegmenter) → Tokenise → Train (weighted loss + label smoothing) → Eval → Push to Hub

In [1]:
!pip install -q transformers datasets evaluate scikit-learn accelerate huggingface_hub underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 56.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but yo

In [2]:
# ============================================================
# LOAD MODEL + LORA — Hỗ trợ resume từ HuggingFace Hub
# ============================================================

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# ── Login HuggingFace ──────────────────────────────────────
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✓ HuggingFace login OK")

✓ HuggingFace login OK


In [3]:
import os, torch

# ── toggle ──────────────────────────────────────────────────
SMOKE_RUN     = False
SMOKE_SAMPLES = 500

# ── model / hub ─────────────────────────────────────────────
MODEL_NAME = "vinai/phobert-large"
HUB_REPO   = "qdovan03/phobert-large-vsmec-5class"
HF_TOKEN   = os.environ.get("HF_TOKEN")

# ── paths / columns ─────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/phobert-large-vsmec"
TEXT_COL   = "Sentence"
LABEL_COL  = "Emotion"

# ── hyperparams ─────────────────────────────────────────────
MAX_LEN         = 128
BATCH_SIZE      = 16    # giảm vì phobert-large lớn hơn (~370M)
EVAL_BATCH      = 32
LR              = 1e-5  # giảm từ 2e-5 để tránh overfit sớm
WEIGHT_DECAY    = 0.05  # tăng từ 0.01 để regularize mạnh hơn
WARMUP_RATIO    = 0.2   # tăng từ 0.1 để warm up chậm hơn
LABEL_SMOOTHING = 0.1
NUM_EPOCHS      = 3 if SMOKE_RUN else 20
EARLY_STOP      = 5     # tăng từ 3 để cho model thêm cơ hội
SEED            = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")
print(f"Model   : {MODEL_NAME}")
print(f"Config  : LR={LR} | WD={WEIGHT_DECAY} | warmup={WARMUP_RATIO} | label_smoothing={LABEL_SMOOTHING}")

Device  : Tesla T4
Model   : vinai/phobert-large
Config  : LR=1e-05 | WD=0.05 | warmup=0.2 | label_smoothing=0.1


In [4]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed,
)
import evaluate
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

set_seed(SEED)

In [5]:
# 5 lớp gộp từ 7 lớp VSMEC
LABEL_NAMES = ["Enjoyment", "Sadness", "Anger", "Surprise", "Other"]
label2id    = {l: i for i, l in enumerate(LABEL_NAMES)}
id2label    = {i: l for l, i in label2id.items()}
NUM_LABELS  = len(LABEL_NAMES)

# bảng gộp nhãn gốc VSMEC -> 5 lớp (khớp với model fusion)
VSMEC_TO_5 = {
    "Enjoyment": "Enjoyment",
    "Sadness":   "Sadness",
    "Fear":      "Sadness",    # Fear -> Sadness
    "Anger":     "Anger",
    "Disgust":   "Anger",      # Disgust -> Anger
    "Surprise":  "Surprise",
    "Other":     "Other",
}

print(f"Labels ({NUM_LABELS}):")
for name, idx in label2id.items():
    print(f"  {idx}  {name}")

Labels (5):
  0  Enjoyment
  1  Sadness
  2  Anger
  3  Surprise
  4  Other


In [6]:
raw = load_dataset("tridm/UIT-VSMEC")
print(raw)

# GỘP nhãn gốc 7 -> 5 lớp NGAY (ghi đè cột Emotion)
def merge_to_5(batch):
    batch[LABEL_COL] = [VSMEC_TO_5[l] for l in batch[LABEL_COL]]
    return batch
raw = raw.map(merge_to_5, batched=True, desc="merge 7->5")

# giờ check mới đúng (chỉ còn 5 nhãn)
unexpected = set(raw["train"].unique(LABEL_COL)) - set(LABEL_NAMES)
assert not unexpected, f"Unexpected labels: {unexpected}"

full_train_label_ids = [label2id[l] for l in raw["train"][LABEL_COL]]

if SMOKE_RUN:
    raw["train"]      = raw["train"].shuffle(seed=SEED).select(range(SMOKE_SAMPLES))
    raw["validation"] = raw["validation"].shuffle(seed=SEED).select(range(100))
    raw["test"]       = raw["test"].shuffle(seed=SEED).select(range(100))

print(f"\nSizes → train: {len(raw['train'])}  val: {len(raw['validation'])}  test: {len(raw['test'])}")

README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

valid.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5548 [00:00<?, ? examples/s]

...

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 5548
    })
    validation: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 686
    })
    test: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 693
    })
})


merge 7->5:   0%|          | 0/5548 [00:00<?, ? examples/s]

merge 7->5:   0%|          | 0/686 [00:00<?, ? examples/s]

merge 7->5:   0%|          | 0/693 [00:00<?, ? examples/s]


Sizes → train: 5548  val: 686  test: 693


In [7]:
# ── Back-translation augmentation (VI → EN → VI) ─────────────────────────
# Chỉ augment train set, chỉ các class ít mẫu (class_weight > 1.5)
# Dùng Helsinki-NLP/opus-mt — free, offline, chạy trên Kaggle T4
# Tắt nếu SMOKE_RUN để tiết kiệm thời gian

AUGMENT = not SMOKE_RUN
AUG_CLASSES   = ["Anger", "Surprise", "Sadness"]   # các lớp ít mẫu sau khi gộp 5
AUG_TIMES     = 2                                # augment 2x mỗi sample

if AUGMENT:
    from transformers import MarianMTModel, MarianTokenizer
    import gc

    VI2EN = "Helsinki-NLP/opus-mt-vi-en"
    EN2VI = "Helsinki-NLP/opus-mt-en-vi"

    print("Loading translation models...")
    tok_vi2en = MarianTokenizer.from_pretrained(VI2EN)
    mdl_vi2en = MarianMTModel.from_pretrained(VI2EN).to(device)
    tok_en2vi = MarianTokenizer.from_pretrained(EN2VI)
    mdl_en2vi = MarianMTModel.from_pretrained(EN2VI).to(device)
    mdl_vi2en.eval()
    mdl_en2vi.eval()

    def translate_batch(texts, tokenizer, model, max_len=128, batch_size=64):
        results = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_len).to(device)
            with torch.no_grad():
                output_ids = model.generate(**inputs, num_beams=4, max_length=max_len)
            results.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
        return results

    def back_translate(texts):
        en = translate_batch(texts, tok_vi2en, mdl_vi2en)
        vi = translate_batch(en,    tok_en2vi, mdl_en2vi)
        return vi

    # Lấy các sample cần augment
    train_df = raw["train"].to_pandas()
    aug_rows  = []

    for label in AUG_CLASSES:
        subset = train_df[train_df[LABEL_COL] == label][TEXT_COL].tolist()
        print(f"  Augmenting {label}: {len(subset)} samples × {AUG_TIMES}")
        for _ in range(AUG_TIMES):
            augmented = back_translate(subset)
            for text in augmented:
                aug_rows.append({TEXT_COL: text, LABEL_COL: label})

    # Ghép vào train set
    import pandas as pd
    from datasets import Dataset, concatenate_datasets

    aug_dataset = Dataset.from_pandas(pd.DataFrame(aug_rows))
    raw["train"] = concatenate_datasets([raw["train"], aug_dataset])
    raw["train"] = raw["train"].shuffle(seed=SEED)

    # Giải phóng VRAM trước khi train
    del mdl_vi2en, mdl_en2vi
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\nTrain size sau augmentation: {len(raw['train'])} (từ {len(train_df)})")

    # Cập nhật class weights theo distribution mới
    full_train_label_ids = [label2id[l] for l in raw["train"][LABEL_COL]]
else:
    print("Augmentation skipped (SMOKE_RUN=True)")

Loading translation models...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

  Augmenting Anger: 1462 samples × 2
  Augmenting Surprise: 242 samples × 2
  Augmenting Sadness: 1265 samples × 2

Train size sau augmentation: 11486 (từ 5548)


In [8]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_LABELS),
    y=full_train_label_ids,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("Class weights (balanced):")
for name, w in zip(LABEL_NAMES, class_weights):
    bar = "█" * int(w * 8)
    print(f"  {name:<12} {w:.4f}  {bar}")

Class weights (balanced):
  Enjoyment    1.4745  ███████████
  Sadness      0.6053  ████
  Anger        0.5238  ████
  Surprise     3.1642  █████████████████████████
  Other        2.2500  █████████████████


In [9]:
# ── normalize + word-segment cho PhoBERT ──────────────────────────────────
# PhoBERT dùng BPE trên word-segmented text → BẮT BUỘC phải segment trước tokenize.
# underthesea.word_tokenize trả về list tokens → join bằng space.
import re
from underthesea import word_tokenize

_WS_RE     = re.compile(r"\s+")
_REPEAT_RE = re.compile(r"(.)\1{2,}")

def normalize_text(text: str) -> str:
    text = text.strip()
    text = _REPEAT_RE.sub(r"\1\1", text)
    text = _WS_RE.sub(" ", text)
    return text

def preprocess(batch):
    normalized = [normalize_text(t) for t in batch[TEXT_COL]]
    # word_tokenize trả về list[str] → join thành string
    batch[TEXT_COL] = [" ".join(word_tokenize(t)) for t in normalized]
    return batch

raw = raw.map(preprocess, batched=True, desc="normalize+segment")
print("Sample sau normalize+segment:", raw["train"][0][TEXT_COL])

normalize+segment:   0%|          | 0/11486 [00:00<?, ? examples/s]

normalize+segment:   0%|          | 0/686 [00:00<?, ? examples/s]

normalize+segment:   0%|          | 0/693 [00:00<?, ? examples/s]

Sample sau normalize+segment: cuộc sống mà .. ghi không được thì thôi . sao mấy bạn chửi người ta dữ vậy


In [10]:
# ── encode string labels → integer ids ──────────────────────
def encode_labels(batch):
    batch["labels"] = [label2id[l] for l in batch[LABEL_COL]]
    return batch

raw = raw.map(encode_labels, batched=True)

# ── tokenise ────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,      # dynamic padding → DataCollatorWithPadding
    )

tokenized = raw.map(
    tokenize,
    batched=True,
    remove_columns=[TEXT_COL, LABEL_COL],
)
tokenized.set_format("torch")
collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample = tokenized["train"][0]
print("Sample keys :", list(sample.keys()))
print("input_ids   :", sample["input_ids"][:10], "...")
print("label       :", sample["labels"].item(), "→", id2label[sample["labels"].item()])

Map:   0%|          | 0/11486 [00:00<?, ? examples/s]

Map:   0%|          | 0/686 [00:00<?, ? examples/s]

Map:   0%|          | 0/693 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/11486 [00:00<?, ? examples/s]

Map:   0%|          | 0/686 [00:00<?, ? examples/s]

Map:   0%|          | 0/693 [00:00<?, ? examples/s]

Sample keys : ['labels', 'input_ids', 'attention_mask']
input_ids   : tensor([   0,  110,  235,   64, 3457,  701,   17,   11,   54,  886]) ...
label       : 4 → Other


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

total_params  = sum(p.numel() for p in model.parameters())
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params / 1e6:.1f} M")
print(f"Trainable params : {train_params / 1e6:.1f} M")

pytorch_model.bin:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Total params     : 369.2 M
Trainable params : 369.2 M


In [12]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor | None = None,
                 label_smoothing: float = 0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights   = class_weights      # stored on CPU, moved per step
        self._label_smoothing = label_smoothing    # áp trực tiếp trong loss

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # **kwargs hứng extra args của Transformers mới (e.g. num_items_in_batch)
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        weights = (
            self._class_weights.to(logits.device)
            if self._class_weights is not None
            else None
        )
        # NOTE: vì ta override compute_loss, label_smoothing_factor trong
        # TrainingArguments sẽ KHÔNG có tác dụng → phải truyền vào đây.
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=weights,
            label_smoothing=self._label_smoothing,
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [13]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc  = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1_w = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    f1_m = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_weighted": f1_w, "f1_macro": f1_m}

In [14]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── schedule ────────────────────────────────────────────
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    # NOTE: label_smoothing_factor KHÔNG set ở đây — ta override compute_loss
    #       nên nó sẽ bị bỏ qua. Label smoothing được áp trong WeightedTrainer.

    # ── precision ───────────────────────────────────────────
    fp16=True,

    # ── eval / checkpoint ───────────────────────────────────
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",   # data lệch mạnh → ưu tiên macro-F1
    greater_is_better=True,
    save_total_limit=2,         # giữ tối đa 2 checkpoint → tiết kiệm disk

    # ── logging ─────────────────────────────────────────────
    logging_steps=20,
    report_to="none",

    # ── seed ────────────────────────────────────────────────
    seed=SEED,
    data_seed=SEED,

    # ── hub (dùng khi push_to_hub thủ công cuối notebook) ──
    hub_model_id=HUB_REPO,
    hub_token=HF_TOKEN,
    push_to_hub=False,
)

print("Training args OK")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args OK


In [15]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,   # Transformers ≥ 4.47
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP)],
    class_weights=class_weights_tensor,
    label_smoothing=LABEL_SMOOTHING,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,1.638605,1.609253,0.272595,0.286657,0.245140
2,1.327777,1.218054,0.539359,0.538535,0.501548
3,1.118594,1.081419,0.581633,0.585470,0.564466
4,1.072374,1.084599,0.620991,0.616838,0.596407
5,0.894068,1.151376,0.637026,0.630464,0.619656
6,0.818519,1.129070,0.670554,0.673621,0.651805
7,0.795442,1.193052,0.645773,0.644753,0.619821
8,0.701938,1.261974,0.655977,0.658293,0.647458
9,0.663696,1.312387,0.657434,0.659117,0.631962
10,0.632626,1.356611,0.669096,0.665728,0.653449


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=6103, training_loss=0.8243787663306796, metrics={'train_runtime': 5728.3154, 'train_samples_per_second': 40.103, 'train_steps_per_second': 1.253, 'total_flos': 2.220511340739905e+16, 'train_loss': 0.8243787663306796, 'epoch': 17.0})

In [20]:
trainer.save_model(OUTPUT_DIR)
# hoặc push lên Hub
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/qdovan03/phobert-large-vsmec-5class/commit/be47d3076f1ef6f8c73bae55517717bd46298363', commit_message='End of training', commit_description='', oid='be47d3076f1ef6f8c73bae55517717bd46298363', pr_url=None, repo_url=RepoUrl('https://huggingface.co/qdovan03/phobert-large-vsmec-5class', endpoint='https://huggingface.co', repo_type='model', repo_id='qdovan03/phobert-large-vsmec-5class'), pr_revision=None, pr_num=None)

In [16]:
test_out = trainer.predict(tokenized["test"])
pred_ids = np.argmax(test_out.predictions, axis=-1)
true_ids = test_out.label_ids

wf1 = f1_score(true_ids, pred_ids, average="weighted")
print(f"Weighted F1 (test): {wf1:.4f}\n")
print(classification_report(true_ids, pred_ids, target_names=LABEL_NAMES, digits=4))

Weighted F1 (test): 0.6762

              precision    recall  f1-score   support

   Enjoyment     0.7094    0.7461    0.7273       193
     Sadness     0.6905    0.7160    0.7030       162
       Anger     0.6994    0.7035    0.7014       172
    Surprise     0.6571    0.6216    0.6389        37
       Other     0.5789    0.5116    0.5432       129

    accuracy                         0.6782       693
   macro avg     0.6671    0.6598    0.6628       693
weighted avg     0.6754    0.6782    0.6762       693



## 13 · Push to Hub
Chỉ chạy khi `SMOKE_RUN = False`.

In [17]:
from huggingface_hub import login
login(token=HF_TOKEN)

trainer.push_to_hub(commit_message="PhoBERT-large fine-tuned on UIT-VSMEC")
tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/qdovan03/phobert-large-vsmec-5class/commit/92f91949ebaa099e00d3e03421e5dfe46eb650f4', commit_message='Upload tokenizer', commit_description='', oid='92f91949ebaa099e00d3e03421e5dfe46eb650f4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/qdovan03/phobert-large-vsmec-5class', endpoint='https://huggingface.co', repo_type='model', repo_id='qdovan03/phobert-large-vsmec-5class'), pr_revision=None, pr_num=None)